In [0]:
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah AS

SELECT
    c.patient_id,
    c.service_date,
    master.latest_insurance_type,
    master.patient_age,
    master.severity,
    pt.territory_id,
    pt.territory_name,
    tr.region_id,
    tr.region_name

FROM (

    SELECT DISTINCT
        PATIENT_ID AS patient_id,
        SERVICE_DATE AS service_date
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 = '8497600101'
      AND service_date >= DATE('2026-03-01')

    UNION ALL

    SELECT DISTINCT
        PATIENT_ID AS patient_id,
        FILL_DATE AS service_date
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 = '8497600101'
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE >= DATE('2026-03-01')

    UNION ALL

    SELECT DISTINCT
        PATIENT_ID AS patient_id,
        SERVICE_DATE AS service_date
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN ('J3490', 'J3590', 'J9999')
      AND SERVICE_DATE >= DATE('2026-03-01')
) c

-- INNER JOIN to patient360_master, filtered to AVLAYAH only
-- This restricts the cohort to patients flagged as Avlayah in patient360
INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master master
    ON c.patient_id = master.patient_id
   AND master.latest_treatment_type = 'AVLAYAH'

LEFT JOIN (
    SELECT DISTINCT
        patient_id,
        primary_hcp_territory_id_2yr AS territory_id,
        primary_hcp_territory_2yr AS territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_territory_id_2yr IS NOT NULL
) pt
    ON c.patient_id = pt.patient_id

LEFT JOIN (
    SELECT
        territory_id,
        MAX(region_id) AS region_id,
        MAX(region_name) AS region_name
    FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
    GROUP BY territory_id
) tr
    ON pt.territory_id = tr.territory_id;


In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
where territory_name is not null

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah AS

SELECT DISTINCT
    d.crx_patient_id AS patient_id,
    CAST(d.ship_date AS DATE) AS service_date,
    z.region_id,
    z.region_name,
    z.territory_id,
    z.territory_name
FROM com_edp_prd.com_intgr.sp_dispense d
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
    ON REGEXP_EXTRACT(d.shipment_hcp_zip, '^[0-9]{5}', 0) = CAST(z.zipcode AS STRING)
WHERE d.ndc = '84976-0001-01'
  AND UPPER(d.fill_type) = 'PAID'
  AND d.returned_flag = 'N'

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah AS (
 
  WITH occam_status AS (
    SELECT DISTINCT
        s.crx_patient_id AS patient_id,
        CAST(s.status_datetime AS DATE) AS service_date
    FROM com_edp_prd.com_intgr.sp_status s
    WHERE UPPER(TRIM(s.status_source)) ilike '%OCCAM%'
      AND s.crx_patient_id IS NOT NULL
      AND s.status_datetime IS NOT NULL
  )
 
  SELECT DISTINCT
      o.patient_id,
      o.service_date,
      z.region_id,
      z.region_name,
      z.territory_id,
      z.territory_name
 
  FROM occam_status o
 
  LEFT JOIN com_edp_prd.com_intgr.sp_patients p
      ON o.patient_id = p.crx_patient_id
 
  LEFT JOIN com_edp_prd.com_intgr.sp_hcp h
      ON p.current_crx_hcp_id = h.crx_hcp_id
 
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
      ON REGEXP_EXTRACT(
            COALESCE(h.hcp_address_zip_postal_code, h.`hcp_address_zip_postal_code`),
            '^[0-9]{5}', 0
         ) = CAST(z.zipcode AS STRING)
 
);

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah AS (

  SELECT
      z.region_id,
      z.region_name,
      z.territory_id,
      z.territory_name,
      DATE(a.modified_date__v) AS service_date,   
      SUM(COALESCE(a.dnli_tivi_patients__c,0)) AS crm_patient_count

  FROM com_edp_prd.com_raw.vcrm_account__v a

  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
    ON LPAD(SUBSTRING(CAST(a.postal_code_cda__v AS STRING), 1, 5), 5, '0')
       =
       LPAD(CAST(z.zipcode AS STRING), 5, '0')

  WHERE a.ispersonaccount__v = false   
    -- AND a.npi__v IS NOT NULL
    AND a.dnli_tivi_patients__c IS NOT NULL

  GROUP BY
      z.region_id,
      z.region_name,
      z.territory_id,
      z.territory_name,
      DATE(a.modified_date__v)
)

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.hcp_prescribed_avlayah AS

WITH base_patients AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

all_hcp_events AS (

    -- MEDICAL NDC
    SELECT DISTINCT
        p.patient_id,
        m.service_date,
        TRIM(COALESCE(m.rendering_npi, m.referring_npi)) AS npi
    FROM base_patients p
    INNER JOIN com_edp_prd.com_raw.kom_medical_events m
        ON p.patient_id = m.patient_id
    WHERE TRIM(m.ndc11) = '8497600101'
      AND COALESCE(m.rendering_npi, m.referring_npi) IS NOT NULL
      AND TRIM(COALESCE(m.rendering_npi, m.referring_npi)) <> ''

    UNION ALL

    -- PHARMACY NDC
    SELECT DISTINCT
        p.patient_id,
        r.fill_date AS service_date,
        TRIM(r.prescriber_npi) AS npi
    FROM base_patients p
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events r
        ON p.patient_id = r.patient_id
    WHERE TRIM(r.ndc11) = '8497600101'
      AND UPPER(TRIM(r.transaction_result)) = 'PAID'
      AND r.prescriber_npi IS NOT NULL
      AND TRIM(r.prescriber_npi) <> ''

    UNION ALL

    -- PROCEDURE CODE
    SELECT DISTINCT
        p.patient_id,
        m.service_date,
        TRIM(COALESCE(m.rendering_npi, m.referring_npi)) AS npi
    FROM base_patients p
    INNER JOIN com_edp_prd.com_raw.kom_medical_events m
        ON p.patient_id = m.patient_id
    WHERE TRIM(m.procedure_code) IN ('J3490', 'J3590', 'J9999')
      AND COALESCE(m.rendering_npi, m.referring_npi) IS NOT NULL
      AND TRIM(COALESCE(m.rendering_npi, m.referring_npi)) <> ''
      AND m.service_date >= DATE('2026-03-01')
),

hcp_geo AS (
    SELECT DISTINCT
        e.service_date,
        e.npi,
        z.region_id,
        z.region_name,
        z.territory_id,
        z.territory_name
    FROM all_hcp_events e
    LEFT JOIN com_edp_prd.com_raw.kom_providers kp
        ON TRIM(e.npi) = TRIM(kp.npi)
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON LEFT(TRIM(kp.provider_zip), 5) = LEFT(TRIM(CAST(z.zipcode AS STRING)), 5)
    WHERE e.npi IS NOT NULL
      AND z.territory_id IS NOT NULL
)

SELECT
    service_date,
    npi,
    region_id,
    region_name,
    territory_id,
    territory_name
FROM hcp_geo;

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah AS
 
-- WITH accounts_dedup AS (
--     SELECT *
--     FROM (
--         SELECT *,
--                ROW_NUMBER() OVER (
--                    PARTITION BY crx_account_id
--                    ORDER BY ingestion_date DESC
--                ) AS rn
--         FROM com_edp_prd.com_intgr.distribution_accounts
--         WHERE is_current = true
--     )
--     WHERE rn = 1
-- ),
 
-- shipments_dedup AS (
--     SELECT *
--     FROM (
--         SELECT *,
--                ROW_NUMBER() OVER (
--                    PARTITION BY invoice_number
--                    ORDER BY ingestion_date DESC
--                ) AS rn
--         FROM com_edp_prd.com_intgr.distribution_sd_shipments
--         WHERE ndc = '84976-0001-01'
--           AND is_current = true
--     )
--     WHERE rn = 1
-- ),
 
-- zip_map AS (
--     SELECT DISTINCT
--         LEFT(REGEXP_REPLACE(zipcode, '[^0-9]', ''), 5) AS zip5,
--         region_id,
--         region_name,
--         territory_id,
--         territory_name
--     FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
-- ),
 
-- base AS (
--     SELECT *
--     FROM (
--         SELECT
--             CAST(s.order_date AS DATE) AS service_date,
--             s.crx_account_id,
 
--             CASE
--                 WHEN UPPER(a.account_type) = 'SP'
--                      OR UPPER(a.account_facility_name) LIKE '%ORSINI%'
--                 THEN 'SP'
--                 ELSE 'HCO'
--             END AS channel,
 
--             z.region_id,
--             z.region_name,
--             z.territory_id,
--             z.territory_name,
 
--             ROW_NUMBER() OVER (
--                 PARTITION BY s.crx_account_id
--                 ORDER BY CAST(s.order_date AS DATE) ASC, s.invoice_number ASC
--             ) AS rn
 
--         FROM shipments_dedup s
--         LEFT JOIN accounts_dedup a
--             ON s.crx_account_id = a.crx_account_id
--         LEFT JOIN zip_map z
--             ON LEFT(REGEXP_REPLACE(s.ship_to_address_postal_code, '[^0-9]', ''), 5) = z.zip5
 
--         WHERE z.territory_id IS NOT NULL
--     )
--     WHERE rn = 1
-- )
 
-- SELECT
--     service_date,
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
 
--     COUNT(DISTINCT crx_account_id) AS hco_accounts_ordered
 
-- FROM base
-- WHERE channel = 'HCO'
 
-- GROUP BY
--     service_date,
--     region_id,
--     region_name,
--     territory_id,
--     territory_name;
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah AS

SELECT
    CURRENT_DATE() AS service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    COUNT(DISTINCT crx_account_id) AS hco_accounts_ordered
FROM com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level
WHERE account_type = 'HCO'
  AND LTD > 0
GROUP BY
    region_id,
    region_name,
    territory_id,
    territory_name;

In [0]:
Select * from com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah 

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.sp_vials_dispensed_avlayah AS

WITH zip_map AS (

    SELECT DISTINCT
        LEFT(REGEXP_REPLACE(zipcode,'[^0-9]',''),5) AS zip5,
        region_id,
        region_name,
        territory_id,
        territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping

),

sp_dispense_dedup AS (

    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY crx_shipment_id
                   ORDER BY ingestion_date DESC
               ) rn 
        FROM com_edp_prd.com_intgr.sp_dispense
        WHERE is_current = true
          AND ndc = '84976-0001-01'
    )
    WHERE rn = 1

),

base_sp_dispense AS (

    SELECT
        CAST(ship_date AS DATE) AS service_date,
        z.region_id,
        z.region_name,
        z.territory_id,
        z.territory_name,
        shipped_quantity
    FROM sp_dispense_dedup d
    LEFT JOIN zip_map z
        ON LEFT(REGEXP_REPLACE(d.shipment_hcp_zip,'[^0-9]',''),5) = z.zip5
    WHERE returned_flag = 'N'
      AND z.territory_id IS NOT NULL

)

SELECT
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    SUM(shipped_quantity) AS SP_Vials_Dispensed
FROM base_sp_dispense
GROUP BY
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name;

In [0]:
select sum(SP_Vials_Dispensed) from com_edp_prd.cmpa_insights_internal_schema.sp_vials_dispensed_avlayah

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.vials_avlayah AS
 
WITH accounts_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY crx_account_id
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_accounts
        WHERE is_current = true
    )
    WHERE rn = 1
),
 
shipments_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY invoice_number
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_sd_shipments
        WHERE ndc = '84976-0001-01'
          AND is_current = true
    )
    WHERE rn = 1
),
 
zip_map AS (
    SELECT DISTINCT
        REGEXP_REPLACE(
    LEFT(REGEXP_REPLACE(zipcode, '[^0-9]', ''), 5),
    '^0+',
    ''
    ) AS zip5,
        region_id,
        region_name,
        territory_id,
        territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
),
 
base_shipments AS (
    SELECT
        CAST(s.order_date AS DATE) AS service_date,
        s.crx_account_id,
        s.quantity_shipped,
        CASE
            WHEN UPPER(a.account_type) = 'SP'
                 OR UPPER(a.account_facility_name) LIKE '%ORSINI%'
            THEN 'SP'
            ELSE 'HCO'
        END AS final_channel,
 
        z.region_id,
        z.region_name,
        z.territory_id,
        z.territory_name
    FROM shipments_dedup s
    LEFT JOIN accounts_dedup a
        ON s.crx_account_id = a.crx_account_id
    LEFT JOIN zip_map z
        ON REGEXP_REPLACE(
    LEFT(REGEXP_REPLACE(s.ship_to_address_postal_code, '[^0-9]', ''), 5),
    '^0+',
    ''
) = z.zip5
    WHERE z.territory_id IS NOT NULL
),
 
vials_agg AS (
    SELECT
        service_date,
        region_id,
        region_name,
        territory_id,
        territory_name,
 
        SUM(quantity_shipped) AS total_vials,
 
        SUM(CASE
                WHEN final_channel = 'SP' THEN quantity_shipped
                ELSE 0
            END) AS sp_vials,
 
        SUM(CASE
                WHEN final_channel = 'HCO' THEN quantity_shipped
                ELSE 0
            END) AS hco_vials
    FROM base_shipments
    GROUP BY
        service_date,
        region_id,
        region_name,
        territory_id,
        territory_name
)
 
SELECT
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    total_vials,
    sp_vials,
    hco_vials,
    ROUND(CASE WHEN total_vials > 0 THEN sp_vials * 1.0 / total_vials ELSE 0 END, 4) AS sp_vials_pct,
    ROUND(CASE WHEN total_vials > 0 THEN hco_vials * 1.0 / total_vials ELSE 0 END, 4) AS hco_vials_pct
FROM vials_agg;


In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.vials_avlayah

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah AS (

-- CLAIMS
SELECT 
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    'CLAIMS_PATIENTS' AS metric_name,
    COALESCE(COUNT(DISTINCT patient_id),0) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
GROUP BY 1,2,3,4,5,6

UNION ALL

-- SP
SELECT 
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    'SP_PATIENTS',
    COALESCE(COUNT(DISTINCT patient_id),0)
FROM com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah
GROUP BY 1,2,3,4,5,6

UNION ALL

-- HUB
SELECT 
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    'HUB_PATIENTS',
    COALESCE(COUNT(DISTINCT patient_id),0)
FROM com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah
GROUP BY 1,2,3,4,5,6

UNION ALL

-- CRM (NO DISTINCT)
SELECT 
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    'CRM_PATIENTS',
    COALESCE(SUM(crm_patient_count),0)
FROM com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah
GROUP BY 1,2,3,4,5,6

);

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period_insurance AS (

-- LTD
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    CASE 
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICAID%' THEN 'Medicaid'
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICARE%' THEN 'Medicare'
        WHEN UPPER(latest_insurance_type) LIKE '%COMMERCIAL%' THEN 'Commercial'
        ELSE 'Unknown'
    END AS insurance_group,
    'LTD' AS period_type,
    'PATIENT_COUNT' AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
WHERE service_date >= DATE('2026-03-01')
GROUP BY 1,2,3,4,5,6,7

UNION ALL

-- YTD
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    CASE 
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICAID%' THEN 'Medicaid'
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICARE%' THEN 'Medicare'
        WHEN UPPER(latest_insurance_type) LIKE '%COMMERCIAL%' THEN 'Commercial'
        ELSE 'Unknown'
    END AS insurance_group,
    'YTD' AS period_type,
    'PATIENT_COUNT' AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
GROUP BY 1,2,3,4,5,6,7

UNION ALL

-- QTD
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    CASE 
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICAID%' THEN 'Medicaid'
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICARE%' THEN 'Medicare'
        WHEN UPPER(latest_insurance_type) LIKE '%COMMERCIAL%' THEN 'Commercial'
        ELSE 'Unknown'
    END AS insurance_group,
    'QTD' AS period_type,
    'PATIENT_COUNT' AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
GROUP BY 1,2,3,4,5,6,7

UNION ALL

-- MTD
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    CASE 
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICAID%' THEN 'Medicaid'
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICARE%' THEN 'Medicare'
        WHEN UPPER(latest_insurance_type) LIKE '%COMMERCIAL%' THEN 'Commercial'
        ELSE 'Unknown'
    END AS insurance_group,
    'MTD' AS period_type,
    'PATIENT_COUNT' AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
GROUP BY 1,2,3,4,5,6,7

);

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period AS

WITH runtime AS (
    SELECT
        CURRENT_DATE() AS end_date,
        DATE('2026-03-01') AS ltd_start,
        DATE('2023-08-01') AS ltd_start_Elaprase,
        DATE_TRUNC('year', CURRENT_DATE()) AS ytd_start,
        DATE_TRUNC('quarter', CURRENT_DATE()) AS qtd_start,
        DATE_TRUNC('month', CURRENT_DATE()) AS mtd_start
),

periods AS (
    SELECT 'LTD' period_type, ltd_start start_date FROM runtime
    UNION ALL
    SELECT 'YTD', ytd_start FROM runtime
    UNION ALL
    SELECT 'QTD', qtd_start FROM runtime
    UNION ALL
    SELECT 'MTD', mtd_start FROM runtime
),

periods_elaprase AS (
    SELECT 'LTD' period_type, ltd_start_elaprase start_date FROM runtime
    UNION ALL
    SELECT 'YTD', ytd_start FROM runtime
    UNION ALL
    SELECT 'QTD', qtd_start FROM runtime
    UNION ALL
    SELECT 'MTD', mtd_start FROM runtime
),

-- ============================================
-- DIMENSIONS
-- ============================================
dim_geo AS (

    SELECT DISTINCT
        region_id,
        region_name,
        territory_id,
        territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah

    UNION

    SELECT
        CAST(NULL AS INT) AS region_id,
        CAST(NULL AS STRING) AS region_name,
        CAST(NULL AS INT) AS territory_id,
        CAST(NULL AS STRING) AS territory_name
),

dim_metrics AS (
    SELECT DISTINCT metric_name
    FROM (

        SELECT metric_name
        FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah

        UNION ALL SELECT 'AVLAYAH_Medicare_Patients'
        UNION ALL SELECT 'AVLAYAH_Medicaid_Patients'
        UNION ALL SELECT 'AVLAYAH_Commercial_Patients'
        UNION ALL SELECT 'AVLAYAH_Unknown_Patients'

        UNION ALL SELECT 'AVLAYAH_PATIENTS_LT_5'
        UNION ALL SELECT 'AVLAYAH_PATIENTS_5_10'
        UNION ALL SELECT 'AVLAYAH_PATIENTS_11_16'
        UNION ALL SELECT 'AVLAYAH_PATIENTS_17_PLUS'

        UNION ALL SELECT 'Severe_Avlayah'
        UNION ALL SELECT 'Attenuated_Avlayah'

        UNION ALL SELECT 'ELAPRASE_LT_5'
        UNION ALL SELECT 'ELAPRASE_5_10'
        UNION ALL SELECT 'ELAPRASE_11_16'
        UNION ALL SELECT 'ELAPRASE_17_PLUS'

        UNION ALL SELECT 'MPSII_PATIENTS_LT_5'
        UNION ALL SELECT 'MPSII_PATIENTS_5_10'
        UNION ALL SELECT 'MPSII_PATIENTS_11_16'
        UNION ALL SELECT 'MPSII_PATIENTS_17_PLUS'

        UNION ALL SELECT 'HCP_PRESCRIBED'
        UNION ALL SELECT 'HCO_ORDERED'

        UNION ALL SELECT 'TOTAL_VIALS_ORDERED'
        UNION ALL SELECT 'TOTAL_VIALS_DISPENSED'
        UNION ALL SELECT 'SP_VIALS_ORDERED'
        UNION ALL SELECT 'SP_VIALS_DISPENSED'
        UNION ALL SELECT 'HCO_VIALS'

        UNION ALL SELECT 'Avlayah_Claims'
        UNION ALL SELECT 'Avlayah_SP'
        UNION ALL SELECT 'Avlayah_HUB'
        UNION ALL SELECT 'Avlayah_CRM'
    )
),

-- ============================================
-- DEDUP CRM
-- ============================================
crm_dedup AS (
    SELECT
        region_id,
        region_name,
        territory_id,
        territory_name,
        service_date,
        SUM(crm_patient_count) AS crm_patient_count
    FROM com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah
    GROUP BY 1,2,3,4,5
),

claims_base AS (
    SELECT *
    FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
),

fact_data AS (

-- ============================================
-- ALL_METRICS_AVLAYAH
-- ============================================
SELECT
    a.region_id,
    a.region_name,
    a.territory_id,
    a.territory_name,
    p.period_type,
    a.metric_name,
    SUM(a.metric_value) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah a
JOIN periods p
    ON p.start_date IS NOT NULL
   AND a.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- INSURANCE KPIs
-- ============================================
SELECT
    c.region_id,
    c.region_name,
    c.territory_id,
    c.territory_name,
    p.period_type,
    CASE
        WHEN latest_insurance_type ILIKE '%MEDICARE%'
            THEN 'AVLAYAH_Medicare_Patients'
        WHEN latest_insurance_type ILIKE '%MEDICAID%'
            THEN 'AVLAYAH_Medicaid_Patients'
        WHEN latest_insurance_type ILIKE '%COMMERCIAL%'
            THEN 'AVLAYAH_Commercial_Patients'
        ELSE 'AVLAYAH_Unknown_Patients'
    END AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM claims_base c
JOIN periods p
    ON p.start_date IS NOT NULL
   AND c.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- AGE KPIs
-- ============================================
SELECT
    c.region_id,
    c.region_name,
    c.territory_id,
    c.territory_name,
    p.period_type,
    CASE
        WHEN patient_age < 5 THEN 'AVLAYAH_PATIENTS_LT_5'
        WHEN patient_age BETWEEN 5 AND 10 THEN 'AVLAYAH_PATIENTS_5_10'
        WHEN patient_age BETWEEN 11 AND 16 THEN 'AVLAYAH_PATIENTS_11_16'
        ELSE 'AVLAYAH_PATIENTS_17_PLUS'
    END AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM claims_base c
JOIN periods p
    ON p.start_date IS NOT NULL
   AND c.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- SEVERITY KPIs
-- ============================================
SELECT
    c.region_id,
    c.region_name,
    c.territory_id,
    c.territory_name,
    p.period_type,
    CASE
        WHEN severity = 'Severe'
            THEN 'Severe_Avlayah'
        ELSE 'Attenuated_Avlayah'
    END AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM claims_base c
JOIN periods p
    ON p.start_date IS NOT NULL
   AND c.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- ELAPRASE KPIs
-- ============================================
SELECT
    COALESCE(a.primary_hcp_region_id_2yr,-1) AS region_id,
    COALESCE(a.primary_hcp_region_2yr,'UNKNOWN') AS region_name,
    COALESCE(a.primary_hcp_territory_id_2yr,-1) AS territory_id,
    COALESCE(a.primary_hcp_territory_2yr,'UNKNOWN') AS territory_name,
    p.period_type,
    CASE
        WHEN a.patient_age < 5 THEN 'ELAPRASE_LT_5'
        WHEN a.patient_age BETWEEN 5 AND 10 THEN 'ELAPRASE_5_10'
        WHEN a.patient_age BETWEEN 11 AND 16 THEN 'ELAPRASE_11_16'
        ELSE 'ELAPRASE_17_PLUS'
    END AS metric_name,
    COUNT(DISTINCT a.patient_id) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
CROSS JOIN periods_elaprase p
WHERE a.first_incidence_treatment_date IS NOT NULL
  AND COALESCE(UPPER(a.latest_treatment_type), '') <> 'AVLAYAH'
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    COALESCE(a.primary_hcp_region_id_2yr, -1) AS region_id,
    COALESCE(a.primary_hcp_region_2yr, 'UNKNOWN') AS region_name,
    COALESCE(a.primary_hcp_territory_id_2yr, -1) AS territory_id,
    COALESCE(a.primary_hcp_territory_2yr, 'UNKNOWN') AS territory_name,
    p.period_type,

    CASE
        WHEN a.patient_age < 5
            THEN 'MPSII_PATIENTS_LT_5'

        WHEN a.patient_age BETWEEN 5 AND 10
            THEN 'MPSII_PATIENTS_5_10'

        WHEN a.patient_age BETWEEN 11 AND 16
            THEN 'MPSII_PATIENTS_11_16'

        ELSE 'MPSII_PATIENTS_17_PLUS'
    END AS metric_name,

    COUNT(DISTINCT a.patient_id) AS metric_value

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a

CROSS JOIN periods p

GROUP BY
    COALESCE(a.primary_hcp_region_id_2yr, -1),
    COALESCE(a.primary_hcp_region_2yr, 'UNKNOWN'),
    COALESCE(a.primary_hcp_territory_id_2yr, -1),
    COALESCE(a.primary_hcp_territory_2yr, 'UNKNOWN'),
    p.period_type,

    CASE
        WHEN a.patient_age < 5
            THEN 'MPSII_PATIENTS_LT_5'

        WHEN a.patient_age BETWEEN 5 AND 10
            THEN 'MPSII_PATIENTS_5_10'

        WHEN a.patient_age BETWEEN 11 AND 16
            THEN 'MPSII_PATIENTS_11_16'

        ELSE 'MPSII_PATIENTS_17_PLUS'
    END

UNION ALL

-- ============================================
-- HCP PRESCRIBED
-- ============================================
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'HCP_PRESCRIBED',
    COUNT(DISTINCT npi)
FROM com_edp_prd.cmpa_insights_internal_schema.hcp_prescribed_avlayah h
JOIN periods p
    ON p.start_date IS NOT NULL
   AND h.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- HCO ORDERED
-- ============================================
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'HCO_ORDERED',
    SUM(hco_accounts_ordered)
FROM com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah h
JOIN periods p
    ON p.start_date IS NOT NULL
   AND h.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- VIALS
-- ============================================
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'TOTAL_VIALS_ORDERED',
    SUM(total_vials)
FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah v
JOIN periods p
    ON p.start_date IS NOT NULL
   AND v.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    COALESCE(h.region_id,s.region_id) region_id,
    COALESCE(h.region_name,s.region_name) region_name,
    COALESCE(h.territory_id,s.territory_id) territory_id,
    COALESCE(h.territory_name,s.territory_name) territory_name,
    COALESCE(h.period_type,s.period_type) period_type,
    'TOTAL_VIALS_DISPENSED' metric_name,
    COALESCE(h.hco_vials,0) + COALESCE(s.sp_vials_dispensed,0) metric_value
FROM (

    SELECT
        region_id,
        region_name,
        territory_id,
        territory_name,
        p.period_type,
        SUM(hco_vials) hco_vials
    FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah v
    JOIN periods p
        ON p.start_date IS NOT NULL
       AND v.service_date >= p.start_date
    GROUP BY 1,2,3,4,5

) h

FULL OUTER JOIN (

    SELECT
        region_id,
        region_name,
        territory_id,
        territory_name,
        p.period_type,
        SUM(sp_vials_dispensed) sp_vials_dispensed
    FROM com_edp_prd.cmpa_insights_internal_schema.sp_vials_dispensed_avlayah v
    JOIN periods p
        ON p.start_date IS NOT NULL
       AND v.service_date >= p.start_date
    GROUP BY 1,2,3,4,5

) s

ON h.region_id = s.region_id
AND h.territory_id = s.territory_id
AND h.period_type = s.period_type

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'SP_VIALS_ORDERED',
    SUM(sp_vials)
FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah v
JOIN periods p
    ON p.start_date IS NOT NULL
   AND v.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'SP_VIALS_DISPENSED',
    SUM(SP_Vials_Dispensed)
FROM com_edp_prd.cmpa_insights_internal_schema.sp_vials_dispensed_avlayah v
JOIN periods p
    ON p.start_date IS NOT NULL
   AND v.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'HCO_VIALS',
    SUM(hco_vials)
FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah v
JOIN periods p
    ON p.start_date IS NOT NULL
   AND v.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- CLAIMS / SP / HUB / CRM
-- ============================================
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'Avlayah_Claims',
    COUNT(DISTINCT patient_id)
FROM claims_base c
JOIN periods p
    ON p.start_date IS NOT NULL
   AND c.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'Avlayah_SP',
    COUNT(DISTINCT patient_id)
FROM com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah s
JOIN periods p
    ON p.start_date IS NOT NULL
   AND s.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'Avlayah_HUB',
    COUNT(DISTINCT patient_id)
FROM com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah h
JOIN periods p
    ON p.start_date IS NOT NULL
   AND h.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'Avlayah_CRM',
    SUM(crm_patient_count)
FROM crm_dedup c
JOIN periods p
    ON p.start_date IS NOT NULL
   AND c.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6
)

-- ============================================
-- FINAL GRID
-- ============================================
SELECT
    g.region_id,
    g.region_name,
    g.territory_id,
    g.territory_name,
    p.period_type,
    m.metric_name,
    COALESCE(SUM(f.metric_value),0) AS metric_value

FROM dim_geo g
CROSS JOIN dim_metrics m
CROSS JOIN periods p

LEFT JOIN fact_data f
    ON g.region_id = f.region_id
   AND g.territory_id = f.territory_id
   AND m.metric_name = f.metric_name
   AND p.period_type = f.period_type

GROUP BY 1,2,3,4,5,6;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period AS
select
'ACCOUNT_LEADS' as dashboard_name,
'OVERVIEW' as section_name,
* FROM com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period

In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period

In [0]:
SELECT
    period_type,
    metric_name,
    SUM(metric_value) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period
WHERE metric_name IN (
    'MPSII_PATIENTS_LT_5',
    'MPSII_PATIENTS_5_10',
    'MPSII_PATIENTS_11_16',
    'MPSII_PATIENTS_17_PLUS'
)
GROUP BY
    period_type,
    metric_name
ORDER BY
    period_type,
    metric_name;

In [0]:
WITH runtime AS (
    SELECT
        end_date,
        DATE('2026-03-01') AS ltd_start,
        DATE_TRUNC('year', end_date) AS ytd_start,
        DATE_TRUNC('quarter', end_date) AS qtd_start,
        DATE_TRUNC('month', end_date) AS mtd_start
    FROM runtime_parameters
),
periods AS (
    SELECT 'LTD' period_type, ltd_start start_date FROM runtime
    UNION ALL
    SELECT 'YTD', ytd_start FROM runtime
    UNION ALL
    SELECT 'QTD', qtd_start FROM runtime
    UNION ALL
    SELECT 'MTD', mtd_start FROM runtime
)
SELECT
    p.period_type,
    SUM(v.sp_vials_dispensed) AS vials
FROM com_edp_prd.cmpa_insights_internal_schema.sp_vials_dispensed_avlayah v
JOIN periods p
    ON v.service_date >= p.start_date
WHERE territory_id = 2006
GROUP BY 1
ORDER BY 1;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level AS

WITH accounts_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY crx_account_id
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_accounts
        WHERE is_current = true
    )
    WHERE rn = 1
),

shipments_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY invoice_number
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_sd_shipments
        WHERE ndc = '84976-0001-01'
          AND is_current = true
    )
    WHERE rn = 1
),
current_data_date AS (
    SELECT
        MAX(CAST(order_date AS DATE)) AS current_wtd_end_date
    FROM shipments_dedup
),
zip_map AS (
    SELECT DISTINCT
        REGEXP_REPLACE(
            LEFT(REGEXP_REPLACE(zipcode, '[^0-9]', ''), 5),
            '^0+',
            ''
        ) AS zip5,
        state,
        region_id,
        region_name,
        territory_id,
        territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
),
date_params AS (
    SELECT
        -- Most recent completed Saturday
        DATE_SUB(
            CURRENT_DATE(),
            CASE
                WHEN DAYOFWEEK(CURRENT_DATE()) = 7 THEN 7
                ELSE DAYOFWEEK(CURRENT_DATE())
            END
        ) AS r1m_end_date,

        DATE_SUB(
            DATE_SUB(
                CURRENT_DATE(),
                CASE
                    WHEN DAYOFWEEK(CURRENT_DATE()) = 7 THEN 7
                    ELSE DAYOFWEEK(CURRENT_DATE())
                END
            ),
            29
        ) AS r1m_start_date,

        -- Rolling 3 Month (90 day) window
        DATE_SUB(
            CURRENT_DATE(),
            CASE
                WHEN DAYOFWEEK(CURRENT_DATE()) = 7 THEN 7
                ELSE DAYOFWEEK(CURRENT_DATE())
            END
        ) AS r3m_end_date,

        DATE_SUB(
            DATE_SUB(
                CURRENT_DATE(),
                CASE
                    WHEN DAYOFWEEK(CURRENT_DATE()) = 7 THEN 7
                    ELSE DAYOFWEEK(CURRENT_DATE())
                END
            ),
            89
        ) AS r3m_start_date
),
base AS (
    SELECT
        s.crx_account_id,
        NULLIF(TRIM(s.crx_account_name), '') AS account_facility_name,
        NULLIF(TRIM(s.ship_to_address_city), '') AS ship_to_address_city,
        a.account_type,
        CAST(s.order_date AS DATE) AS order_date,
        s.quantity_shipped,

        REGEXP_REPLACE(
            LEFT(REGEXP_REPLACE(s.ship_to_address_postal_code, '[^0-9]', ''), 5),
            '^0+',
            ''
        ) AS postal_code,

        z.state,
        z.territory_id,
        z.territory_name,
        z.region_id,
        z.region_name
    FROM shipments_dedup s
    LEFT JOIN accounts_dedup a
        ON s.crx_account_id = a.crx_account_id
    LEFT JOIN zip_map z
        ON REGEXP_REPLACE(
               LEFT(REGEXP_REPLACE(s.ship_to_address_postal_code, '[^0-9]', ''), 5),
               '^0+',
               ''
           ) = z.zip5
)

SELECT
    b.crx_account_id,
    b.account_facility_name,
    b.ship_to_address_city,

    CASE
        WHEN UPPER(b.account_type) = 'SP'
          OR UPPER(b.account_facility_name) LIKE '%ORSINI%'
        THEN 'SP'
        ELSE 'HCO'
    END AS account_type,

    b.postal_code,
    b.state,
    b.territory_id,
    b.territory_name,
    b.region_id,
    b.region_name,

    SUM(
        CASE
            WHEN b.order_date >= DATE('2026-03-01')
            THEN b.quantity_shipped
            ELSE 0
        END
    ) AS LTD,

    SUM(
        CASE
            WHEN b.order_date >= DATE_TRUNC('month', CURRENT_DATE)
            THEN b.quantity_shipped
            ELSE 0
        END
    ) AS MTD,

    SUM(
        CASE
            WHEN b.order_date >= DATE_TRUNC('quarter', CURRENT_DATE)
            THEN b.quantity_shipped
            ELSE 0
        END
    ) AS QTD,

    SUM(
        CASE
            WHEN b.order_date >= DATE_TRUNC('year', CURRENT_DATE)
            THEN b.quantity_shipped
            ELSE 0
        END
    ) AS YTD,

 ------------------------------------------------------------------
-- LAST WEEK (Previous Sunday -> Previous Saturday)
------------------------------------------------------------------

DATE_SUB(CURRENT_DATE(), DAYOFWEEK(CURRENT_DATE()) + 6)
    AS LAST_WEEK_START_DATE,

DATE_SUB(CURRENT_DATE(), DAYOFWEEK(CURRENT_DATE()))
    AS LAST_WEEK_END_DATE,

SUM(
    CASE
        WHEN b.order_date BETWEEN
             DATE_SUB(CURRENT_DATE(), DAYOFWEEK(CURRENT_DATE()) + 6)
         AND DATE_SUB(CURRENT_DATE(), DAYOFWEEK(CURRENT_DATE()))
        THEN b.quantity_shipped
        ELSE 0
    END
) AS LAST_WEEK,

------------------------------------------------------------------
-- CURRENT WTD (Most Recent Sunday -> Latest Date Available)
------------------------------------------------------------------

DATE_SUB(
    cdd.current_wtd_end_date,
    DAYOFWEEK(cdd.current_wtd_end_date) - 1
) AS WTD_START_DATE,

DATE_SUB(CURRENT_DATE(), 1) AS WTD_END_DATE,

SUM(
    CASE
        WHEN b.order_date BETWEEN
             DATE_SUB(
                 cdd.current_wtd_end_date,
                 DAYOFWEEK(cdd.current_wtd_end_date) - 1
             )
         AND cdd.current_wtd_end_date
        THEN b.quantity_shipped
        ELSE 0
    END
) AS WTD,

    dp.r1m_start_date AS R1M_START_DATE,
    dp.r1m_end_date AS R1M_END_DATE,

    SUM(
        CASE
            WHEN b.order_date BETWEEN dp.r1m_start_date
                                  AND dp.r1m_end_date
            THEN b.quantity_shipped
            ELSE 0
        END
    ) AS R1M,
        dp.r3m_start_date AS R3M_START_DATE,
    dp.r3m_end_date AS R3M_END_DATE,

    SUM(
        CASE
            WHEN b.order_date BETWEEN dp.r3m_start_date
                                  AND dp.r3m_end_date
            THEN b.quantity_shipped
            ELSE 0
        END
    ) AS R3M

FROM base b
CROSS JOIN date_params dp
CROSS JOIN current_data_date cdd

GROUP BY
    b.crx_account_id,
    b.account_facility_name,
    b.ship_to_address_city,
    CASE
        WHEN UPPER(b.account_type) = 'SP'
          OR UPPER(b.account_facility_name) LIKE '%ORSINI%'
        THEN 'SP'
        ELSE 'HCO'
    END,
    b.postal_code,
    b.state,
    b.territory_id,
    b.territory_name,
    b.region_id,
    b.region_name,
    dp.r1m_start_date,
    dp.r1m_end_date,
    dp.r3m_start_date,
    dp.r3m_end_date,
    cdd.current_wtd_end_date
    order by LTD desc;

In [0]:
select * from cmpa_insights_internal_schema.hco_ordered_avlayah

In [0]:
Select * from com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level

In [0]:
select * from com_edp_prd.com_raw.kom_providers
where PROVIDER_ZIP = '90095'
-- and PROVIDER_ADDRESS ILIKE '%200%'

In [0]:
select count(account_type) from com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level
where account_type = 'HCO'
and account_facility_name is not null

In [0]:
SELECT DISTINCT
    patient_id,
    prescriber_npi AS npi,
    fill_date AS avlayah_ndc_date,
    'PHARMACY_NDC' AS avlayah_ndc_source
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE ndc11 = '84976000101'
  AND fill_date <= DATE '2026-06-30'

UNION ALL

SELECT DISTINCT
    patient_id,
    COALESCE(rendering_npi, referring_npi, billing_npi) AS npi,
    service_date AS avlayah_ndc_date,
    'MEDICAL_NDC' AS avlayah_ndc_source
FROM com_edp_prd.com_raw.kom_medical_events
WHERE ndc11 = '84976000101'
  AND service_date <= DATE '2026-06-30';

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
where zipcode = '90095'